In [6]:
import sys, os
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed

repo_path = os.path.abspath('~/ukbb-pulmonary-artery/DeepCMR')
assert os.path.isdir(repo_path)
if not repo_path in sys.path: sys.path.append(repo_path)

from utils.strings import str2
import utils.strings as dstr

### Define folders

In [7]:
# ================================== EDIT THESE ==================================
# Location of our project folder
PROJ_ROOT = '{deepcmr_data_root}/nnUNET/'

# Input niftis
input_niftis_root = '../cmr_lvot_20212_niftis/'

# Output files for predicting by nUNet
output_niftis_root = 'exported_nifts_for_predicting/cmr_lvot_20212_niftis/'
# ================================================================================

In [11]:
# Create list of Niftis found in the data folder
PatientNames = [ dstr.standardize_patient_name(str2(i).basename().rchop('.nii.gz'))
                    for i in Path(PROJ_ROOT, input_niftis_root).glob('*.nii.gz') if not str(i).endswith('_gt.nii.gz') ]

print("Number of patients for predicting: %.d" % len(PatientNames))

Number of patients for predicting: 44003


## Generate prediction data files

Create an index and a function that will be paralellized.

In [13]:
output_path = os.path.join(PROJ_ROOT, output_niftis_root)
os.makedirs(output_path, exist_ok=True)

# create the filenames using an index (this part can't be parallelized because the index is serial)
names = []
inputfiles = []
outputpaths = []
outputfilenames = []
max_files_per_directory = 32000
use_batches = (len(PatientNames) * 50 > max_files_per_directory)
if use_batches: print("Using batches of %.d files per folder to mitigate I/O bottlenecks" % max_files_per_directory)
i = 0
for name in tqdm(PatientNames, desc="Indexing patients"):
    names += [name]
    inputfiles += [ str(Path(PROJ_ROOT, input_niftis_root, name + '.nii.gz')) ]
    outpath = os.path.join(output_path, 'batch_%.3d' % int(i/max_files_per_directory)) if use_batches else output_path
    os.makedirs(outpath, exist_ok=True)
    outputpaths += [ outpath ]
    outputfilenames += [ dstr.format_patient_filename(prefix='pa', patient_name=name, frame_num=0, index=i) ]
    i += 50

# define parallel method
def do_extraction(i, input_files, output_paths, output_files):
    V_nifti = nib.load( input_files[i] )            
    V = V_nifti.get_fdata()
    prefix, name, frame, index = dstr.deconstruct_patient_filename( output_files[i] )
    for t in range(50):
        fname = dstr.format_patient_filename(prefix=prefix, patient_name=name, frame_num=frame+t, index=index+t)
        nib.Nifti1Image(V[:,:,:,t], V_nifti.affine).to_filename( os.path.join(output_paths[i], fname + '_0000.nii.gz') )

Using batches of 32000 files per folder to mitigate I/O bottlenecks
Indexing patients: 100%|██████████| 44003/44003 [00:02<00:00, 14958.75it/s]


Split the names into segments of 500 each, and run each segment in parallel with 16 cores.

In [ ]:
# run
num_cores = 16
pts_per_epoch = 500
accumulator = 0
while accumulator < len(names):
    max_index = min(len(names), accumulator + pts_per_epoch)
    names_i = names[accumulator:max_index]
    inputfiles_i = inputfiles[accumulator:max_index]
    outputpaths_i = outputpaths[accumulator:max_index]
    outputfilenames_i = outputfilenames[accumulator:max_index]

    print("\nProcessing patients %.d through %.d, out of %.d total" % (accumulator+1, max_index, len(names)))
    Parallel(n_jobs=num_cores)(delayed(do_extraction)(i, inputfiles_i, outputpaths_i, outputfilenames_i)
        for i in tqdm(range(max_index-accumulator), unit=" patients", desc=("Extracting niftis using %.d cores" % num_cores)))

    accumulator += pts_per_epoch